<a href="https://colab.research.google.com/github/youngbean0519/Self-Project/blob/main/notebooks/05_linear_svm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [61]:
!git clone https://github.com/youngbean0519/Self-Project.git
%pwd

Cloning into 'Self-Project'...
remote: Enumerating objects: 67, done.
remote: Counting objects: 100% (67/67), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 67 (delta 20), reused 53 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (67/67), 480.36 KiB | 4.41 MiB/s, done.
Resolving deltas: 100% (20/20), done.


'/content/Self-Project/Self-Project'

In [33]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [34]:
%cd Self-Project/

/content/Self-Project/Self-Project


In [35]:
import os
import sys

PROJECT_DIR = "/content/Self-Project"
SRC_DIR = os.path.join(PROJECT_DIR, "src")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")
DATA_DIR = "/content/drive/MyDrive/26_KOTE/data"
ARTIFACT_DIR = "/content/drive/MyDrive/26_KOTE/artifacts/week5_svm"

sys.path.append(PROJECT_DIR)
sys.path.append(SRC_DIR)

print("PROJECT_DIR:", PROJECT_DIR)
print("SRC_DIR:", SRC_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("DATA_DIR:", DATA_DIR)
print("ARTIFACT_DIR:", ARTIFACT_DIR)

PROJECT_DIR: /content/Self-Project
SRC_DIR: /content/Self-Project/src
RESULTS_DIR: /content/Self-Project/results
DATA_DIR: /content/drive/MyDrive/26_KOTE/data
ARTIFACT_DIR: /content/drive/MyDrive/26_KOTE/artifacts/week5_svm


In [36]:
import preprocessing.text as text_preprocessing
import preprocessing.labels as preprocessing_labels
import evaluation.metrics as metrics
import utils.labels as utils_labels

In [37]:
import time
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import (
    f1_score,
    hamming_loss,
    average_precision_score
)

In [38]:
train_path = os.path.join(DATA_DIR, "train.tsv")
val_path = os.path.join(DATA_DIR, "val.tsv")
test_path = os.path.join(DATA_DIR, "test.tsv")

train_df = pd.read_csv(train_path, sep="\t", header=None, names=["id", "text", "labels"])
val_df = pd.read_csv(val_path, sep="\t", header=None, names=["id", "text", "labels"])
test_df = pd.read_csv(test_path, sep="\t", header=None, names=["id", "text", "labels"])

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

train: (40000, 3)
val: (5000, 3)
test: (5000, 3)


In [39]:
NUM_LABELS = 44

train_label_lists = train_df["labels"].apply(preprocessing_labels.parse_label_ids).tolist()
val_label_lists = val_df["labels"].apply(preprocessing_labels.parse_label_ids).tolist()
test_label_lists = test_df["labels"].apply(preprocessing_labels.parse_label_ids).tolist()

y_train, mlb = preprocessing_labels.encode_multilabels(train_label_lists, NUM_LABELS)
y_val = mlb.transform(val_label_lists)
y_test = mlb.transform(test_label_lists)

print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

y_train: (40000, 44)
y_val: (5000, 44)
y_test: (5000, 44)


In [40]:
def make_preprocessed_texts(df, normalize_repeat, remove_symbols):
  return df["text"].astype(str).apply(
      lambda text: text_preprocessing.preprocess_text(
          text,
          normalize_repeat=normalize_repeat,
          remove_symbols=remove_symbols
      )
  )

preprocessing_conditions = {
    "raw": {
        "normalize_repeat": False,
        "remove_symbols": False
    },

    "repeat": {
        "normalize_repeat": True,
        "remove_symbols": False
    },

    "repeat_symbols": {
        "normalize_repeat": True,
        "remove_symbols": True
    }
}

In [41]:
sample = train_df["text"].astype(str)

sample_repeat = sample.apply(
    lambda x: text_preprocessing.preprocess_text(
        x,
        normalize_repeat=True,
        remove_symbols=False
    )
)

sample_repeat_symbols = sample.apply(
    lambda x: text_preprocessing.preprocess_text(
        x,
        normalize_repeat=True,
        remove_symbols=True
    )
)

print(
    "raw -> repeat changed:",
    (sample != sample_repeat).sum()
)

print(
    "repeat -> repeat_symbols changed:",
    (sample_repeat != sample_repeat_symbols).sum()
)

print(
    "raw -> repeat_symbols changed:",
    (sample != sample_repeat_symbols).sum()
)

raw -> repeat changed: 13144
repeat -> repeat_symbols changed: 25198
raw -> repeat_symbols changed: 28865


In [42]:
changed_repeat = sample != sample_repeat

for original, processed in zip(
    sample[changed_repeat].head(20),
    sample_repeat[changed_repeat].head(20)
):
    print("ORIGINAL :", original)
    print("REPEAT   :", processed)
    print("-" * 80)

ORIGINAL : 내가 톰행크스를 좋아하긴 했나보다... 초기 영화 빼고는 다 봤네.
REPEAT   : 내가 톰행크스를 좋아하긴 했나보다.. 초기 영화 빼고는 다 봤네.
--------------------------------------------------------------------------------
ORIGINAL : 미역은 원생생물계 산호초는 동물ㅇㅇ 아 미역이 바다의 새ㄱㅇㄱㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ
REPEAT   : 미역은 원생생물계 산호초는 동물ㅇㅇ 아 미역이 바다의 새ㄱㅇㄱㅋㅋ
--------------------------------------------------------------------------------
ORIGINAL : 어릴 때 했던 건데 아직도 볼 때마다 뒤통수 얼얼함 ㅋㅋㅋㅋ
REPEAT   : 어릴 때 했던 건데 아직도 볼 때마다 뒤통수 얼얼함 ㅋㅋ
--------------------------------------------------------------------------------
ORIGINAL : 물갈이약 구매했어요...미미네에서 수조2개랑 여러가지 용품사면서 3~4번 주문했었는데 그 동안 만족했었는데... 약간 불량들이 있네요... 1) 함께 구매한 스포이드에 구멍이 있네요... 2) 부화통에 같이오는 흡착고무가 불량이라 안붙어요... 3) 예전에 구매한 온도계의 온도가 안맞아요...
REPEAT   : 물갈이약 구매했어요..미미네에서 수조2개랑 여러가지 용품사면서 3~4번 주문했었는데 그 동안 만족했었는데.. 약간 불량들이 있네요.. 1) 함께 구매한 스포이드에 구멍이 있네요.. 2) 부화통에 같이오는 흡착고무가 불량이라 안붙어요.. 3) 예전에 구매한 온도계의 온도가 안맞아요..
--------------------------------------------------------------------------------
ORIGINAL : 스토리 담당만 붙여줬으면 갓작품 됬을텐데.....
RE

In [43]:
changed_symbols = (
    sample_repeat != sample_repeat_symbols
)

for before, after in zip(
    sample_repeat[changed_symbols].head(20),
    sample_repeat_symbols[changed_symbols].head(20)
):
    print("BEFORE :", before)
    print("AFTER  :", after)
    print("-" * 80)

BEFORE : 내가 톰행크스를 좋아하긴 했나보다.. 초기 영화 빼고는 다 봤네.
AFTER  : 내가 톰행크스를 좋아하긴 했나보다 초기 영화 빼고는 다 봤네
--------------------------------------------------------------------------------
BEFORE : 정말 상상을 초월하는 무개념 진상들 상대하다 우울증, 공항장애 걸리는 공무원 많아요 ㅠ 생각보다 스트레스 심한 직업군이라는 생각입니다.
AFTER  : 정말 상상을 초월하는 무개념 진상들 상대하다 우울증 공항장애 걸리는 공무원 많아요 ㅠ 생각보다 스트레스 심한 직업군이라는 생각입니다
--------------------------------------------------------------------------------
BEFORE : 새로운 세상과 조우한 자의 어린아이 같은 반응, 어쩌면 회복된 것은 눈이 아닌 순수함일지도 모르겠습니다. 그걸 가능케 한 사랑에 경의를 표합니다.
AFTER  : 새로운 세상과 조우한 자의 어린아이 같은 반응 어쩌면 회복된 것은 눈이 아닌 순수함일지도 모르겠습니다 그걸 가능케 한 사랑에 경의를 표합니다
--------------------------------------------------------------------------------
BEFORE : 물갈이약 구매했어요..미미네에서 수조2개랑 여러가지 용품사면서 3~4번 주문했었는데 그 동안 만족했었는데.. 약간 불량들이 있네요.. 1) 함께 구매한 스포이드에 구멍이 있네요.. 2) 부화통에 같이오는 흡착고무가 불량이라 안붙어요.. 3) 예전에 구매한 온도계의 온도가 안맞아요..
AFTER  : 물갈이약 구매했어요 미미네에서 수조2개랑 여러가지 용품사면서 3~4번 주문했었는데 그 동안 만족했었는데 약간 불량들이 있네요 1 함께 구매한 스포이드에 구멍이 있네요 2 부화통에 같이오는 흡착고무가 불량이라 안붙어요 3 예전에 구매한 온도계의 온도가 안맞아요

In [44]:
TFIDF_PARAMS = {
    "ngram_range": (1, 2),
    "min_df": 2,
    "max_features": 50000,
    "sublinear_tf": True
}

In [45]:
THRESHOLDS = np.arange(
    -1.0,
    1.01,
    0.05
)

print("Number of thresholds:", len(THRESHOLDS))
print(THRESHOLDS)

Number of thresholds: 41
[-1.0000000e+00 -9.5000000e-01 -9.0000000e-01 -8.5000000e-01
 -8.0000000e-01 -7.5000000e-01 -7.0000000e-01 -6.5000000e-01
 -6.0000000e-01 -5.5000000e-01 -5.0000000e-01 -4.5000000e-01
 -4.0000000e-01 -3.5000000e-01 -3.0000000e-01 -2.5000000e-01
 -2.0000000e-01 -1.5000000e-01 -1.0000000e-01 -5.0000000e-02
  8.8817842e-16  5.0000000e-02  1.0000000e-01  1.5000000e-01
  2.0000000e-01  2.5000000e-01  3.0000000e-01  3.5000000e-01
  4.0000000e-01  4.5000000e-01  5.0000000e-01  5.5000000e-01
  6.0000000e-01  6.5000000e-01  7.0000000e-01  7.5000000e-01
  8.0000000e-01  8.5000000e-01  9.0000000e-01  9.5000000e-01
  1.0000000e+00]


In [46]:
def evaluate_threshold(y_true, scores, threshold):
  y_pred = (scores >= threshold).astype(int)

  micro_f1 = f1_score(
      y_true,
      y_pred,
      average="micro",
      zero_division=0
  )

  macroe_f1 = f1_score(
      y_true,
      y_pred,
      average="macro",
      zero_division=0
  )

  hamming = hamming_loss(y_true, y_pred)

  return {
      "threshold": threshold,
      "micro_f1": micro_f1,
      "macro_f1": macroe_f1,
      "hamming_loss": hamming
  }


In [47]:
def search_thresholds(y_true, scores, thresholds):
  results = []

  for threshold in thresholds:
    result = evaluate_threshold(y_true, scores, threshold)

    results.append(result)

  return pd.DataFrame(results)

In [48]:
experiment_results = []

for condition_name, condition in preprocessing_conditions.items():
  print("=" * 80)
  print("PREPROCESSING:", condition_name)
  print("=" * 80)

  # 1. Preprocessing

  train_texts = make_preprocessed_texts(
      train_df,
      normalize_repeat=condition["normalize_repeat"],
      remove_symbols=condition["remove_symbols"]
  )

  val_texts = make_preprocessed_texts(
      val_df,
      normalize_repeat=condition["normalize_repeat"],
      remove_symbols=condition["remove_symbols"]
  )

  # 2. TF-IDF

  vectorizer = TfidfVectorizer(**TFIDF_PARAMS)

  X_train = vectorizer.fit_transform(train_texts)
  X_val = vectorizer.transform(val_texts)

  print("X_train shape:", X_train.shape)
  print("X_val shape:", X_val.shape)
  print("Vocabulary size:", len(vectorizer.vocabulary_))

  # 3. Linear SVM

  model = OneVsRestClassifier(LinearSVC(random_state=42))

  # 4. Training

  train_start = time.perf_counter()
  model.fit(X_train, y_train)
  train_time = (time.perf_counter() - train_start)

  # 5. Validation inference

  inference_start = time.perf_counter()
  val_scores = model.decision_function(X_val)
  inference_time = (time.perf_counter() - inference_start)

  # 6. Threshold search

  threshold_results = search_thresholds(y_val, val_scores, THRESHOLDS)
  best_index = (
      threshold_results["micro_f1"].idxmax()
  )
  best_row = threshold_results.loc[best_index]

  #7. PR-AUC

  pr_auc_micro = average_precision_score(y_val, val_scores, average="micro")
  pr_auc_macro = average_precision_score(y_val, val_scores, average="macro")

  # 8. Store result

  experiment_results.append({
      "preprocessing": condition_name,
      "vectorizer": vectorizer,
      "model": model,
      "val_scores": val_scores,
      "threshold_results": threshold_results,
      "best_threshold": float(best_row["threshold"]),
      "micro_f1": float(best_row["micro_f1"]),
      "macro_f1": float(best_row["macro_f1"]),
      "hamming_loss": float(best_row["hamming_loss"]),
      "pr_auc_micro": float(pr_auc_micro),
      "pr_auc_macro": float(pr_auc_macro),
      "train_time": float(train_time),
      "inference_time": float(inference_time),
      "vocab_size": len(vectorizer.vocabulary_)
  })

  print()
  print("Best threshold:", best_row["threshold"])
  print("Micro F1:", best_row["micro_f1"])
  print("Macro F1:", best_row["macro_f1"])
  print("Hamming loss:", best_row["hamming_loss"])
  print("PR-AUC (micro):", pr_auc_micro)
  print("PR-AUC (macro):", pr_auc_macro)
  print("Train time:", train_time)
  print("Inference time:", inference_time)

  print()

PREPROCESSING: raw
X_train shape: (40000, 50000)
X_val shape: (5000, 50000)
Vocabulary size: 50000

Best threshold: -0.39999999999999947
Micro F1: 0.4855322080606269
Macro F1: 0.3863859211541024
Hamming loss: 0.21723636363636364
PR-AUC (micro): 0.48933846068985276
PR-AUC (macro): 0.37099483966418495
Train time: 14.90339329599965
Inference time: 0.017418119000012666

PREPROCESSING: repeat
X_train shape: (40000, 50000)
X_val shape: (5000, 50000)
Vocabulary size: 50000

Best threshold: -0.39999999999999947
Micro F1: 0.4860143249501858
Macro F1: 0.38747744633346476
Hamming loss: 0.21691363636363636
PR-AUC (micro): 0.49052626183427384
PR-AUC (macro): 0.3720574125345299
Train time: 15.332739289000074
Inference time: 0.01757967399998961

PREPROCESSING: repeat_symbols
X_train shape: (40000, 50000)
X_val shape: (5000, 50000)
Vocabulary size: 50000

Best threshold: -0.39999999999999947
Micro F1: 0.4860143249501858
Macro F1: 0.38747744633346476
Hamming loss: 0.21691363636363636
PR-AUC (micro): 0.

In [49]:
comparison_df = pd.DataFrame([{
    "preprocessing": result["preprocessing"],
    "best_threshold": result["best_threshold"],
    "micro_f1": result["micro_f1"],
    "macro_f1": result["macro_f1"],
    "hamming_loss": result["hamming_loss"],
    "pr_auc_micro": result["pr_auc_micro"],
    "pr_auc_macro": result["pr_auc_macro"],
    "train_time": result["train_time"],
    "inference_time": result["inference_time"],
    "vocab_size": result["vocab_size"]
} for result in experiment_results ])

comparison_df = comparison_df.sort_values(by="micro_f1", ascending=False).reset_index(drop=True)

comparison_df

,preprocessing,best_threshold,micro_f1,macro_f1,hamming_loss,pr_auc_micro,pr_auc_macro,train_time,inference_time,vocab_size
0,repeat,-0.4,0.486014,0.387477,0.216914,0.490526,0.372057,15.332739,0.017580,50000
1,repeat_symbols,-0.4,0.486014,0.387477,0.216914,0.490526,0.372057,13.583908,0.025650,50000
2,raw,-0.4,0.485532,0.386386,0.217236,0.489338,0.370995,14.903393,0.017418,50000


In [50]:
BEST_PREPROCESSING = (
    comparison_df.iloc[0]["preprocessing"]
)

best_result = next(
    result
    for result in experiment_results
    if result["preprocessing"]
    == BEST_PREPROCESSING
)

BEST_THRESHOLD = (
    best_result["best_threshold"]
)

best_model = (
    best_result["model"]
)

best_vectorizer = (
    best_result["vectorizer"]
)

best_val_scores = (
    best_result["val_scores"]
)

print(
    "BEST_PREPROCESSING:",
    BEST_PREPROCESSING
)

print(
    "BEST_THRESHOLD:",
    BEST_THRESHOLD
)

BEST_PREPROCESSING: repeat
BEST_THRESHOLD: -0.39999999999999947


In [51]:
best_val_pred = (
    best_val_scores
    >= BEST_THRESHOLD
).astype(int)

print(
    "prediction shape:",
    best_val_pred.shape
)

prediction shape: (5000, 44)


In [52]:
best_micro_f1 = f1_score(
    y_val,
    best_val_pred,
    average="micro",
    zero_division=0
)

best_macro_f1 = f1_score(
    y_val,
    best_val_pred,
    average="macro",
    zero_division=0
)

best_hamming_loss = hamming_loss(
    y_val,
    best_val_pred
)

best_pr_auc_micro = (
    average_precision_score(
        y_val,
        best_val_scores,
        average="micro"
    )
)

best_pr_auc_macro = (
    average_precision_score(
        y_val,
        best_val_scores,
        average="macro"
    )
)

print(
    "Micro F1:",
    best_micro_f1
)

print(
    "Macro F1:",
    best_macro_f1
)

print(
    "Hamming Loss:",
    best_hamming_loss
)

print(
    "PR-AUC Micro:",
    best_pr_auc_micro
)

print(
    "PR-AUC Macro:",
    best_pr_auc_macro
)

Micro F1: 0.4860143249501858
Macro F1: 0.38747744633346476
Hamming Loss: 0.21691363636363636
PR-AUC Micro: 0.49052626183427384
PR-AUC Macro: 0.3720574125345299


In [53]:
threshold_results_df = (
    best_result["threshold_results"]
)

threshold_results_df

,threshold,micro_f1,macro_f1,hamming_loss
0,-1.000000e+00,0.379071,0.337667,0.529873
1,-9.500000e-01,0.391943,0.344495,0.490805
2,-9.000000e-01,0.403182,0.350965,0.456959
3,-8.500000e-01,0.418405,0.359969,0.416377
4,-8.000000e-01,0.429313,0.366373,0.385909
5,-7.500000e-01,0.440694,0.373307,0.356827
6,-7.000000e-01,0.451235,0.379471,0.329795
7,-6.500000e-01,0.462223,0.385141,0.302923
8,-6.000000e-01,0.470291,0.388512,0.281914
9,-5.500000e-01,0.477151,0.390054,0.262686


In [54]:
threshold_results_df.sort_values(
    by="micro_f1",
    ascending=False
).head(10)

,threshold,micro_f1,macro_f1,hamming_loss
12,-0.40,0.486014,0.387477,0.216914
13,-0.35,0.485749,0.382636,0.204614
11,-0.45,0.484317,0.389913,0.228895
10,-0.50,0.482400,0.391560,0.244568
14,-0.30,0.481788,0.376499,0.195105
9,-0.55,0.477151,0.390054,0.262686
15,-0.25,0.477047,0.368110,0.187555
16,-0.20,0.470961,0.358062,0.180691
8,-0.60,0.470291,0.388512,0.281914
17,-0.15,0.465118,0.348795,0.174632


In [55]:
threshold_results_df[
    [
        "threshold",
        "micro_f1",
        "macro_f1",
        "hamming_loss"
    ]
]

threshold_results_df.sort_values(
    by="macro_f1",
    ascending=False
).head(10)

,threshold,micro_f1,macro_f1,hamming_loss
10,-0.50,0.482400,0.391560,0.244568
9,-0.55,0.477151,0.390054,0.262686
11,-0.45,0.484317,0.389913,0.228895
8,-0.60,0.470291,0.388512,0.281914
12,-0.40,0.486014,0.387477,0.216914
7,-0.65,0.462223,0.385141,0.302923
13,-0.35,0.485749,0.382636,0.204614
6,-0.70,0.451235,0.379471,0.329795
14,-0.30,0.481788,0.376499,0.195105
5,-0.75,0.440694,0.373307,0.356827


In [56]:
label_f1_scores = f1_score(
    y_val,
    best_val_pred,
    average=None,
    zero_division=0
)

label_f1_df = pd.DataFrame({
    "label_id":
        np.arange(NUM_LABELS),

    "label": [
        utils_labels.LABELS[i]
        for i in range(NUM_LABELS)
    ],

    "f1":
        label_f1_scores
})

label_f1_df

,label_id,label,f1
0,0,불평/불만,0.692221
1,1,환영/호의,0.491914
2,2,감동/감탄,0.550435
3,3,지긋지긋,0.340271
4,4,고마움,0.484076
5,5,슬픔,0.325114
6,6,화남/분노,0.623204
7,7,존경,0.388785
8,8,기대감,0.510715
9,9,우쭐댐/무시함,0.289550


In [57]:
train_label_counts = (
    y_train.sum(axis=0)
)

label_f1_df["train_count"] = (
    train_label_counts
)

label_f1_df = label_f1_df[
    [
        "label_id",
        "label",
        "train_count",
        "f1"
    ]
]

label_f1_df

,label_id,label,train_count,f1
0,0,불평/불만,16696,0.692221
1,1,환영/호의,9252,0.491914
2,2,감동/감탄,10778,0.550435
3,3,지긋지긋,6386,0.340271
4,4,고마움,5181,0.484076
5,5,슬픔,4137,0.325114
6,6,화남/분노,12299,0.623204
7,7,존경,3854,0.388785
8,8,기대감,11197,0.510715
9,9,우쭐댐/무시함,6002,0.289550


In [58]:
label_f1_df.sort_values(
    by="f1",
    ascending=False
).head(15)

,label_id,label,train_count,f1
0,0,불평/불만,16696,0.692221
10,10,안타까움/실망,17465,0.662987
22,22,짜증,14930,0.645978
23,23,어이없음,15996,0.644785
6,6,화남/분노,12299,0.623204
28,28,즐거움/신남,10867,0.565295
20,20,한심함,12201,0.564542
2,2,감동/감탄,10778,0.550435
42,42,기쁨,9967,0.550388
12,12,의심/불신,11565,0.528736


In [59]:
label_f1_df.sort_values(
    by="f1",
    ascending=True
).head(15)

,label_id,label,train_count,f1
30,30,죄책감,698,0.034188
17,17,부끄러움,2329,0.104247
18,18,공포/무서움,1194,0.105769
36,36,서러움,2031,0.134615
25,25,패배/자기혐오,1581,0.136808
26,26,귀찮음,2428,0.150735
35,35,부담/안_내킴,4768,0.205167
19,19,절망,3652,0.208068
27,27,힘듦/지침,3769,0.260000
11,11,비장함,3352,0.263811


In [60]:
label_f1_df.sort_values(
    by="train_count",
    ascending=True
).head(15)

,label_id,label,train_count,f1
30,30,죄책감,698,0.034188
18,18,공포/무서움,1194,0.105769
25,25,패배/자기혐오,1581,0.136808
36,36,서러움,2031,0.134615
17,17,부끄러움,2329,0.104247
26,26,귀찮음,2428,0.150735
11,11,비장함,3352,0.263811
19,19,절망,3652,0.208068
37,37,재미없음,3691,0.398779
27,27,힘듦/지침,3769,0.260000


In [63]:
%cd /content/Self-Project/
!git pull origin main

/content/Self-Project
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 5 (delta 1), reused 5 (delta 1), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 1.81 KiB | 617.00 KiB/s, done.
From https://github.com/youngbean0519/Self-Project
 * branch            main       -> FETCH_HEAD
   99716b8..eeb2380  main       -> origin/main
Updating 99716b8..eeb2380
Fast-forward
 results/week04_lr_label_f1.csv                 | 45 ++++++++++++++++++++++++++
 results/week04_lr_preprocessing_comparison.csv |  4 +++
 2 files changed, 49 insertions(+)
 create mode 100644 results/week04_lr_label_f1.csv
 create mode 100644 results/week04_lr_preprocessing_comparison.csv


In [64]:
lr_comparison_path = os.path.join(RESULTS_DIR, "week04_lr_preprocessing_comparison.csv")
lr_label_f1_path = os.path.join(RESULTS_DIR, "week04_lr_label_f1.csv")

lr_comparison_df = pd.read_csv(lr_comparison_path)
lr_label_f1_df = pd.read_csv(lr_label_f1_path)

print("Logistic Regression preprocessing results:")
display(lr_comparison_df)

Logistic Regression preprocessing results:


,preprocessing,threshold,micro_f1,macro_f1,hamming_loss,pr_auc_micro,pr_auc_macro,train_time,inference_time,vocab_size
0,repeat,0.25,0.512276,0.365655,0.200636,0.523327,0.395635,85.492888,0.042816,50000
1,repeat_symbols,0.25,0.512276,0.365655,0.200636,0.523327,0.395635,81.582936,0.034706,50000
2,raw,0.25,0.511358,0.364154,0.201227,0.521789,0.394299,73.131260,0.070128,50000


In [66]:
lr_best = (
    lr_comparison_df
    .sort_values(
        by="micro_f1",
        ascending=False
    )
    .iloc[0]
)

svm_best = (
    comparison_df
    .sort_values(
        by="micro_f1",
        ascending=False
    )
    .iloc[0]
)

model_comparison_df = pd.DataFrame([
    {
        "model":
            "Logistic Regression",

        "preprocessing":
            lr_best["preprocessing"],

        "best_threshold":
            lr_best["threshold"],

        "micro_f1":
            lr_best["micro_f1"],

        "macro_f1":
            lr_best["macro_f1"],

        "hamming_loss":
            lr_best["hamming_loss"],

        "pr_auc_micro":
            lr_best["pr_auc_micro"],

        "pr_auc_macro":
            lr_best["pr_auc_macro"],

        "train_time":
            lr_best["train_time"],

        "inference_time":
            lr_best["inference_time"]
    },

    {
        "model":
            "Linear SVM",

        "preprocessing":
            svm_best["preprocessing"],

        "best_threshold":
            svm_best["best_threshold"],

        "micro_f1":
            svm_best["micro_f1"],

        "macro_f1":
            svm_best["macro_f1"],

        "hamming_loss":
            svm_best["hamming_loss"],

        "pr_auc_micro":
            svm_best["pr_auc_micro"],

        "pr_auc_macro":
            svm_best["pr_auc_macro"],

        "train_time":
            svm_best["train_time"],

        "inference_time":
            svm_best["inference_time"]
    }
])

model_comparison_df

,model,preprocessing,best_threshold,micro_f1,macro_f1,hamming_loss,pr_auc_micro,pr_auc_macro,train_time,inference_time
0,Logistic Regression,repeat,0.25,0.512276,0.365655,0.200636,0.523327,0.395635,85.492888,0.042816
1,Linear SVM,repeat,-0.40,0.486014,0.387477,0.216914,0.490526,0.372057,15.332739,0.017580


In [67]:
lr_label_comparison = (
    lr_label_f1_df[
        [
            "label_id",
            "label",
            "f1"
        ]
    ]
    .rename(
        columns={
            "f1": "lr_f1"
        }
    )
)

svm_label_comparison = (
    label_f1_df[
        [
            "label_id",
            "label",
            "f1"
        ]
    ]
    .rename(
        columns={
            "f1": "svm_f1"
        }
    )
)

label_comparison_df = (
    lr_label_comparison.merge(
        svm_label_comparison,
        on=[
            "label_id",
            "label"
        ]
    )
)

label_comparison_df[
    "svm_minus_lr"
] = (
    label_comparison_df["svm_f1"]
    - label_comparison_df["lr_f1"]
)

label_comparison_df

,label_id,label,lr_f1,svm_f1,svm_minus_lr
0,0,불평/불만,0.674715,0.692221,0.017506
1,1,환영/호의,0.510413,0.491914,-0.018498
2,2,감동/감탄,0.568688,0.550435,-0.018253
3,3,지긋지긋,0.352635,0.340271,-0.012364
4,4,고마움,0.482999,0.484076,0.001077
5,5,슬픔,0.286897,0.325114,0.038218
6,6,화남/분노,0.610465,0.623204,0.012739
7,7,존경,0.311111,0.388785,0.077674
8,8,기대감,0.538815,0.510715,-0.028100
9,9,우쭐댐/무시함,0.265134,0.289550,0.024416


In [69]:
label_comparison_df.sort_values(
    by="svm_minus_lr",
    ascending=False
).head(15)

,label_id,label,lr_f1,svm_f1,svm_minus_lr
19,19,절망,0.096475,0.208068,0.111593
18,18,공포/무서움,0.000000,0.105769,0.105769
25,25,패배/자기혐오,0.031250,0.136808,0.105558
27,27,힘듦/지침,0.162866,0.260000,0.097134
35,35,부담/안_내킴,0.112175,0.205167,0.092992
17,17,부끄러움,0.013699,0.104247,0.090548
26,26,귀찮음,0.065672,0.150735,0.085064
7,7,존경,0.311111,0.388785,0.077674
21,21,역겨움/징그러움,0.208909,0.283220,0.074311
34,34,경악,0.234957,0.309019,0.074062


In [70]:
label_comparison_df.sort_values(
    by="svm_minus_lr",
    ascending=True
).head(15)

,label_id,label,lr_f1,svm_f1,svm_minus_lr
33,33,당황/난처,0.480296,0.433862,-0.046433
15,15,신기함/관심,0.517950,0.487662,-0.030289
39,39,놀람,0.365629,0.337303,-0.028325
8,8,기대감,0.538815,0.510715,-0.028100
40,40,행복,0.508316,0.482236,-0.026080
42,42,기쁨,0.572024,0.550388,-0.021636
12,12,의심/불신,0.547741,0.528736,-0.019005
1,1,환영/호의,0.510413,0.491914,-0.018498
2,2,감동/감탄,0.568688,0.550435,-0.018253
31,31,증오/혐오,0.532455,0.514615,-0.017841


In [71]:
problem_label_ids = [
    30,  # 죄책감
    18,  # 공포/무서움
    17,  # 부끄러움
    25   # 패배/자기혐오
]

label_comparison_df[
    label_comparison_df[
        "label_id"
    ].isin(problem_label_ids)
].sort_values(
    by="label_id"
)

,label_id,label,lr_f1,svm_f1,svm_minus_lr
17,17,부끄러움,0.013699,0.104247,0.090548
18,18,공포/무서움,0.000000,0.105769,0.105769
25,25,패배/자기혐오,0.031250,0.136808,0.105558
30,30,죄책감,0.000000,0.034188,0.034188


In [72]:
error_count = 0

for i in range(len(val_df)):

    true_ids = np.where(
        y_val[i] == 1
    )[0]

    pred_ids = np.where(
        best_val_pred[i] == 1
    )[0]

    if set(true_ids) != set(pred_ids):

        print("TEXT:")
        print(
            val_df.iloc[i]["text"]
        )

        print("TRUE:")
        print([
            utils_labels.LABELS[j]
            for j in true_ids
        ])

        print("PRED:")
        print([
            utils_labels.LABELS[j]
            for j in pred_ids
        ])

        print("-" * 80)

        error_count += 1

        if error_count == 10:
            break

TEXT:
음료가 몇푼이나 한다고.. 미국같으면 바로 바꿔줄텐데 참 어이없네요 그 카페
TRUE:
['불평/불만', '화남/분노', '안타까움/실망', '짜증', '어이없음', '서러움']
PRED:
['불평/불만', '화남/분노', '안타까움/실망', '의심/불신', '한심함', '짜증', '어이없음', '즐거움/신남', '당황/난처', '기쁨']
--------------------------------------------------------------------------------
TEXT:
아버님 끝까지 진실이 밝혀질때까지 응원합니다.
TRUE:
['환영/호의', '감동/감탄', '존경', '기대감', '비장함', '신기함/관심', '아껴주는', '불쌍함/연민', '안심/신뢰']
PRED:
['환영/호의', '감동/감탄', '고마움', '존경', '기대감', '안타까움/실망', '신기함/관심', '아껴주는', '안심/신뢰']
--------------------------------------------------------------------------------
TEXT:
대응이 잘못됐어 차라리 그냥 철없던 시절 그랬었다고 인정하고 울면서 자숙하는 시늉만 했어도 이렇게까지 최악은 아니었을텐데
TRUE:
['불평/불만', '안타까움/실망', '의심/불신', '한심함', '짜증', '어이없음', '증오/혐오', '서러움']
PRED:
['불평/불만', '감동/감탄', '화남/분노', '안타까움/실망', '비장함', '의심/불신', '한심함', '역겨움/징그러움', '짜증', '힘듦/지침', '불안/걱정']
--------------------------------------------------------------------------------
TEXT:
저번주 일요일 지리산둘레길엘 갔다왔습니다. "강호동,은지원 코스" 상상이하!!! 기대가 컸던 만큼 실망도 크더군요.
TRUE:
['불평/불만', '안타까움/실망', '짜증']
PRED:
['

In [73]:
comparison_df.to_csv(
    os.path.join(
        RESULTS_DIR,
        "week05_svm_preprocessing_comparison.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

label_f1_df.to_csv(
    os.path.join(
        RESULTS_DIR,
        "week05_svm_label_f1.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

model_comparison_df.to_csv(
    os.path.join(
        RESULTS_DIR,
        "week05_lr_vs_svm_comparison.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

label_comparison_df.to_csv(
    os.path.join(
        RESULTS_DIR,
        "week05_lr_vs_svm_label_comparison.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

In [74]:
os.makedirs(ARTIFACT_DIR, exist_ok=True)

print("ARTIFACT_DIR:", ARTIFACT_DIR)
print("exists:", os.path.exists(ARTIFACT_DIR))

ARTIFACT_DIR: /content/drive/MyDrive/26_KOTE/artifacts/week5_svm
exists: True


In [75]:
joblib.dump(
    best_model,
    os.path.join(
        ARTIFACT_DIR,
        "linear_svm.joblib"
    )
)

joblib.dump(
    best_vectorizer,
    os.path.join(
        ARTIFACT_DIR,
        "tfidf_vectorizer.joblib"
    )
)

joblib.dump(
    mlb,
    os.path.join(
        ARTIFACT_DIR,
        "multilabel_binarizer.joblib"
    )
)

['/content/drive/MyDrive/26_KOTE/artifacts/week5_svm/multilabel_binarizer.joblib']

In [76]:
best_condition = (
    preprocessing_conditions[
        BEST_PREPROCESSING
    ]
)

config = {
    "model":
        "Linear SVM",

    "vectorizer":
        "TF-IDF",

    "preprocessing":
        BEST_PREPROCESSING,

    "normalize_repeat":
        best_condition[
            "normalize_repeat"
        ],

    "remove_symbols":
        best_condition[
            "remove_symbols"
        ],

    "threshold":
        BEST_THRESHOLD,

    "score_type":
        "decision_function",

    "num_labels":
        NUM_LABELS,

    "tfidf": {
        "ngram_range": [1, 2],
        "min_df": 2,
        "max_features": 50000,
        "sublinear_tf": True
    },

    "svm": {
        "type": "LinearSVC",
        "random_state": 42
    }
}

with open(
    os.path.join(
        ARTIFACT_DIR,
        "config.json"
    ),
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        config,
        f,
        ensure_ascii=False,
        indent=2
    )

In [77]:
print(os.listdir(ARTIFACT_DIR))

['linear_svm.joblib', 'tfidf_vectorizer.joblib', 'multilabel_binarizer.joblib', 'config.json']


In [78]:
print("=" * 80)
print("WEEK 05 - TF-IDF + LINEAR SVM")
print("=" * 80)

print(
    "Best preprocessing :",
    BEST_PREPROCESSING
)

print(
    "Best threshold     :",
    BEST_THRESHOLD
)

print(
    "Micro F1           :",
    round(
        best_micro_f1,
        4
    )
)

print(
    "Macro F1           :",
    round(
        best_macro_f1,
        4
    )
)

print(
    "Hamming Loss       :",
    round(
        best_hamming_loss,
        4
    )
)

print(
    "PR-AUC Micro       :",
    round(
        best_pr_auc_micro,
        4
    )
)

print(
    "PR-AUC Macro       :",
    round(
        best_pr_auc_macro,
        4
    )
)

print(
    "Training Time      :",
    round(
        best_result["train_time"],
        2
    ),
    "sec"
)

print(
    "Inference Time     :",
    round(
        best_result["inference_time"],
        4
    ),
    "sec"
)

print()
print("=" * 80)
print("LR vs SVM")
print("=" * 80)

display(
    model_comparison_df
)

WEEK 05 - TF-IDF + LINEAR SVM
Best preprocessing : repeat
Best threshold     : -0.39999999999999947
Micro F1           : 0.486
Macro F1           : 0.3875
Hamming Loss       : 0.2169
PR-AUC Micro       : 0.4905
PR-AUC Macro       : 0.3721
Training Time      : 15.33 sec
Inference Time     : 0.0176 sec

LR vs SVM


,model,preprocessing,best_threshold,micro_f1,macro_f1,hamming_loss,pr_auc_micro,pr_auc_macro,train_time,inference_time
0,Logistic Regression,repeat,0.25,0.512276,0.365655,0.200636,0.523327,0.395635,85.492888,0.042816
1,Linear SVM,repeat,-0.40,0.486014,0.387477,0.216914,0.490526,0.372057,15.332739,0.017580
